In [1]:
from sklearn.datasets import load_diabetes 
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import numpy as np

In [2]:
X, y = load_diabetes(return_X_y=True)

In [7]:
pipe = Pipeline([
    ("sc", StandardScaler()),
    ("model", XGBRegressor(random_state=42)),
])

param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.01, 0.1],
}

grid = GridSearchCV(pipe, param_grid, scoring="neg_mean_squared_error", cv=5)
grid.fit(X, y)

print("최적 파라미터:", grid.best_params_)
print(f"최적 MSE: {-grid.best_score_:.2f}")
print(f"RMSE:     {np.sqrt(-grid.best_score_):.2f}")

최적 파라미터: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300}
최적 MSE: 3266.24
RMSE:     57.15


In [8]:
best = grid.best_estimator_
r2 = cross_val_score(best, X, y, scoring="r2", cv=5)
print(f"R^2: {r2.mean():.3f}")

R^2: 0.434


## 결론: diabetes에서는 XGBoost < lasso

| 모델 | MSE | R² |
|---|---|---|
| lasso (03) | 2992 | 0.482 |
| XGBoost (06) | 3266 | 0.434 |

- best_params가 가장 단순한 값(max_depth=3, lr=0.01) → 이 데이터엔 XGBoost가 과함
- diabetes는 작고(442개) 선형적 → 트리 부스팅의 강점이 안 통함
- **"표 데이터=XGBoost 최강"은 조건부. 크고 비선형일 때만 성립**
- → 다음: California housing(20,640개)으로 XGBoost의 진가 확인
